In [ ]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS fraud_dataset
OPTIONS(
location="us",
default_table_expiration_days=14 );

Query is running:   0%|          |

""


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_detection (
  type STRING,
  name STRING,
  weight FLOAT64
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery

LOAD DATA into fraud_dataset.fraud_detection
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv'],
  autodetect = true,
  skip_leading_rows = 1
);

Executing query with job ID: 9cc8a33b-1f69-4b4b-9288-575f00f1d1b1
Query executing: 0.18s


ERROR:
 400 unsupported option autodetect; reason: invalidQuery, location: query, message: unsupported option autodetect

Location: US
Job ID: 9cc8a33b-1f69-4b4b-9288-575f00f1d1b1



In [ ]:
%%bigquery
LOAD DATA OVERWRITE fraud_dataset.fraud_detection
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv'],
  skip_leading_rows = 1
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    * EXCEPT(Employment_Status, Device_Type),
    IF(Employment_Status = 'Full-time', 1, 0) AS is_full_time,
    IF(Employment_Status = 'Part-time', 1, 0) AS is_part_time,
    IF(Employment_Status = 'Unemployed', 1, 0) AS is_unemployed,
    IF(Device_Type = 'Mobile', 1, 0) AS is_mobile,
    IF(Device_Type = 'Desktop', 1, 0) AS is_desktop
FROM fraud_dataset.fraud_detection;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    * EXCEPT(age),
    IF(age BETWEEN 18 AND 24, 1, 0) AS age_18_24,
    IF(age BETWEEN 25 AND 34, 1, 0) AS age_25_34,
    IF(age BETWEEN 35 AND 44, 1, 0) AS age_35_44,
    IF(age >= 45, 1, 0) AS age_45_plus
FROM fraud_dataset.fraud_data_modified;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    *,
    SAFE_DIVIDE(Income, Amount_Requested) AS income_to_amount_ratio
FROM fraud_dataset.fraud_data_modified;

Query is running:   0%|          |

""


In [ ]:

%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    *,
    DATE_DIFF(CURRENT_DATE(), last_assistance_date, DAY) AS time_since_previous_assistance
FROM fraud_dataset.fraud_data_modified;

Executing query with job ID: 1ae90875-2430-478e-bccf-83f6354e339a
Query executing: 0.40s


ERROR:
 400 Unrecognized name: last_assistance_date at [4:31]; reason: invalidQuery, location: query, message: Unrecognized name: last_assistance_date at [4:31]

Location: US
Job ID: 1ae90875-2430-478e-bccf-83f6354e339a



In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    *,
    DATE_DIFF(CURRENT_DATE(), CAST(Previous_Assistance_Date AS DATE), DAY) AS time_since_previous_assistance
FROM fraud_dataset.fraud_data_modified;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT column_name
FROM fraud_dataset.INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'fraud_data_modified';

Query is running:   0%|          |

Downloading:   0%|          |

,column_name
0,Applicant_ID
1,Income
2,Number_of_Dependents
3,Amount_Requested
4,Previous_Assistance_Received
5,Previous_Assistance_Date
6,Supporting_Doc_Verified
7,Application_Frequency_Last_Year
8,IP_Address
9,Application_Date


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    * EXCEPT(is_fraud, has_prior_default),
    CAST(is_fraud AS INT64) AS is_fraud_numeric,
    CAST(has_prior_default AS INT64) AS has_prior_default_numeric
FROM fraud_dataset.fraud_data_modified;

Executing query with job ID: 786ffc13-d0fc-4689-bd5b-f81bfab43bcf
Query executing: 0.45s


ERROR:
 400 Column is_fraud in SELECT * EXCEPT list does not exist at [3:14]; reason: invalidQuery, location: query, message: Column is_fraud in SELECT * EXCEPT list does not exist at [3:14]

Location: US
Job ID: 786ffc13-d0fc-4689-bd5b-f81bfab43bcf



In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    * EXCEPT(Previous_Assistance_Received, Supporting_Doc_Verified, Fraudulent),
    CAST(Previous_Assistance_Received AS INT64) AS Previous_Assistance_Received_numeric,
    CAST(Supporting_Doc_Verified AS INT64) AS Supporting_Doc_Verified_numeric,
    CAST(Fraudulent AS INT64) AS Fraudulent_numeric
FROM fraud_dataset.fraud_data_modified;

Executing query with job ID: 36c0b5cb-b433-447d-9400-72efa8adb15b
Query executing: 0.48s


ERROR:
 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/qwiklabs-gcp-01-bb0d31ab0812/jobs/36c0b5cb-b433-447d-9400-72efa8adb15b?projection=full&location=US&prettyPrint=false: API deadline too short

Location: US
Job ID: 36c0b5cb-b433-447d-9400-72efa8adb15b



In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT * EXCEPT(Previous_Assistance_Received),
CAST(Previous_Assistance_Received AS INT64) AS Previous_Assistance_Received_numeric
FROM fraud_dataset.fraud_data_modified;

Executing query with job ID: c779e83e-edd3-4160-954f-1d24bbe4ee65
Query executing: 0.38s


ERROR:
 400 Column Previous_Assistance_Received in SELECT * EXCEPT list does not exist at [2:17]; reason: invalidQuery, location: query, message: Column Previous_Assistance_Received in SELECT * EXCEPT list does not exist at [2:17]

Location: US
Job ID: c779e83e-edd3-4160-954f-1d24bbe4ee65



In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    * EXCEPT(Fraudulent),
    -- Convert the main target column to 0 or 1
    CAST(Fraudulent AS INT64) AS Fraudulent_numeric
FROM fraud_dataset.fraud_data_modified;

Executing query with job ID: 6ec11955-20bc-48f6-a05c-d211382711f5
Query executing: 0.46s


ERROR:
 400 Column Fraudulent in SELECT * EXCEPT list does not exist at [3:14]; reason: invalidQuery, location: query, message: Column Fraudulent in SELECT * EXCEPT list does not exist at [3:14]

Location: US
Job ID: 6ec11955-20bc-48f6-a05c-d211382711f5



In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fraud_dataset.fraud_data_modified AS
SELECT
    * EXCEPT(Employment_Status, Device_Type, age, Previous_Assistance_Received, Supporting_Doc_Verified, Fraudulent),

    -- a. One-hot encode Employment_Status and Device_Type
    IF(Employment_Status = 'Full-time', 1, 0) AS is_full_time,
    IF(Employment_Status = 'Part-time', 1, 0) AS is_part_time,
    IF(Device_Type = 'Mobile', 1, 0) AS is_mobile,
    IF(Device_Type = 'Desktop', 1, 0) AS is_desktop,

    -- b. Bin age and one-hot encode
    IF(age BETWEEN 18 AND 24, 1, 0) AS age_18_24,
    IF(age BETWEEN 25 AND 34, 1, 0) AS age_25_34,
    IF(age >= 35, 1, 0) AS age_35_plus,

    -- c. Income-to-Amount-Requested ratio
    SAFE_DIVIDE(Income, Amount_Requested) AS income_to_amount_ratio,

    -- d. Time Since Previous Assistance
    DATE_DIFF(CURRENT_DATE(), CAST(Previous_Assistance_Date AS DATE), DAY) AS Time_Since_Previous_Assistance,

    -- e. Change True/False fields to 1s and 0s
    CAST(Previous_Assistance_Received AS INT64) AS Previous_Assistance_Received_numeric,
    CAST(Supporting_Doc_Verified AS INT64) AS Supporting_Doc_Verified_numeric,
    CAST(Fraudulent AS INT64) AS Fraudulent_numeric

FROM `fraud_dataset.fraud_detection`;

Query is running:   0%|          |

""
